<a href="https://colab.research.google.com/github/sanjaliroy/berkeley-homes-wildfire-agent-simulation/blob/main/wildfire_transcript_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homeowners Interview Transcript Pre-processing

Course: INFO 290: Fundamentals of Generative AI (Spring 2026)







Creator: Amrita Nambiar

What this notebook covers:
1.   Transcription Cleaning: Filters raw text to isolate homeowner responses.
2.   AI-Driven Extraction: Uses Gemini 2.5 Flash model to transform cleaned transcripts into structured profiles.
3.  Data Export: Saves results to Google Drive as agents_extracted.yaml (profiles) and extraction_debug.jsonl (logs).

Why this matters:


1.   Agent Persona Creation: The extracted profiles form the basis for creating realistic and diverse agent personas in a wildfire mitigation simulation.
2.    Structured Data: Converts unstructured interview data into a structured format that can be easily consumed by simulation models.
3.    Automated Extraction: Leverages Generative AI to automate the extraction process, reducing manual effort and ensuring consistency.



## Install Dependencies

In [1]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 3.3 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata, drive
import google.genai as genai
from pathlib import Path

import re
import json
import yaml
import google.generativeai as genai
from docx import Document
from pathlib import Path
import pandas as pd

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Configuration

In [3]:
# Google Drive folder with interview transcripts
transcripts_folder_path = "/content/drive/MyDrive/INFO 290: Intro to Gen AI/Interview Transcripts"

# Output folder
output_folder_path = "/content/drive/MyDrive/INFO 290: Intro to Gen AI/outputs"

# Output filenames
agents_yaml_filename = "agents_extracted.yaml"
debug_jsonl_filename = "extraction_debug.jsonl"

# Model for extraction
model = "models/gemini-2.5-flash"
max_tokens = 2000

# Interviewer name to filter out from transcripts
interviewer_name = "Amrita"

print("Configuration completed")
print(f"Transcripts folder : {transcripts_folder_path}")
print(f"Output folder      : {output_folder_path}")
print(f"Model              : {model}")

Configuration completed
Transcripts folder : /content/drive/MyDrive/INFO 290: Intro to Gen AI/Interview Transcripts
Output folder      : /content/drive/MyDrive/INFO 290: Intro to Gen AI/outputs
Model              : models/gemini-2.5-flash


## Mount Google Drive & Load API Key

In [4]:
# Mount Google Drive
drive.mount("/content/drive")
print("Google Drive mounted")

# Load API key
wildfire_api_key = userdata.get('wildfire')

if wildfire_api_key:
    # Set it as GOOGLE_API_KEY environment variable
    os.environ["GOOGLE_API_KEY"] = wildfire_api_key
    genai.configure(api_key=wildfire_api_key)
    print("API key loaded.")
else:
    print("API key not found.")

# Verify transcript folder exists
transcript_dir = Path(transcripts_folder_path)
if not transcript_dir.exists():
    print(f"\nTranscript folder not found: {transcripts_folder_path}")
else:
    docx_files = sorted(transcript_dir.glob("*.docx"))
    print(f"\nFound {len(docx_files)} .docx transcript(s):")
    for f in docx_files:
        print(f"   • {f.name}")

Mounted at /content/drive
Google Drive mounted
API key loaded.

Found 11 .docx transcript(s):
   • Beth - Mar 6 at 7_03 PM.docx
   • Charles - Mar 6 at 8_06 PM.docx
   • Edward - Mar 11 at 2_06 PM.docx
   • Isabelle G - Feb 02 2026, 5_00 PM.docx
   • Jennifer - Feb 25 at 1_03 PM.docx
   • Lola - Feb 23 at 5_09 PM.docx
   • Michel_.docx
   • Ross - Feb 25 at 10_21 AM.docx
   • Steven L - Feb 02 2026, 12_00 PM.docx
   • Susan_.docx
   • Yen_.docx


## Helper Functions

In [5]:
# Extract plain text from .docx
def extract_text_from_docx(docx_path: Path) -> str:
    # Read all paragraphs from a .docx file and return as plain text
    doc = Document(str(docx_path))
    paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    return "\n\n".join(paragraphs)

In [6]:
# Filter out interviewer turns
def filter_to_interviewee_only(transcript: str, interviewer_name: str = "Amrita") -> str:
    """
    For timestamped transcripts,strip out all interviewer turns. Return homeowner speech only.
    For unformatted transcripts, return full text unchanged.
    """
    # Detect if transcript uses speaker-timestamp format
    if not re.search(r'\w[\w\s]+:\s+\d{2}:\d{2}', transcript):
        return transcript  # if unformatted, return as it is

    lines = transcript.split('\n')
    filtered = []
    include = True

    for line in lines:
        speaker_match = re.match(r'^([\w][\w\s]+):\s+\d{2}:\d{2}', line)
        if speaker_match:
            speaker = speaker_match.group(1).strip()
            include = interviewer_name not in speaker
        if include:
            filtered.append(line)

    return '\n'.join(filtered)


In [7]:
# Extraction prompt
extraction_prompt = extraction_prompt = """\
You are a research assistant helping build realistic agent personas for a wildfire \
mitigation simulation set in the Berkeley Hills, California.

Below is an interview transcript with a Berkeley Hills homeowner. Your job is to \
extract a structured profile that will be used to initialise a simulation agent. \
Second person ("You are...")

TRANSCRIPT:
{transcript}

Extract the following and respond ONLY with valid JSON (no markdown, no preamble, \
no trailing text):

{{
  "agent_id": "<firstname_lowercase — e.g. ross, yen, beth>",
  "display_name": "<A privacy-preserving pseudonym (not the real name). Use a plausible first name that \
matches the person's approximate age, gender, and cultural background — e.g. 'Margaret' for an older \
white woman, 'Diego' for a Latino man in his 40s>",
  "age": "<age or age range — e.g. 35, or null if not mentioned>",
  "occupation": "<their occupation or profession, or null if not mentioned>",
  "marital_status": "<married | single | divorced | widowed | partnered | null>",
  "tenure_type": "<homeowner | renter | null>",
  "location": "<Berkeley Hills (Berkeley Woods Firewise area) | specific neighborhood or general area, or null>",
  "persona": "<3-5 sentence paragraph in third person capturing: who they are, \
how long they have lived there, their attitude toward fire risk, their relationship \
with the Ember ordinance and defensible space requirements, their trust in government \
and fire institutions, and any distinctive personality traits revealed in the interview>",
  "risk_zone": "<high | medium | low — infer from location: ridge/hilltop=high, \
mid-hill=medium, flatter/lower=low, or null if location is insufficient to infer>",
  "tilden_proximity": "<front_line | mid_hills | moderate_distance | far | null>",
  "property_type": "<single_family | rental | residential | null>",
  "lot_size": "<large | standard | small | null>",
  "roof_type": "<tile | composition_shingle | metal | clay | slate | concrete | wood | asphalt | null>",
  "siding": "<stucco | wood | fiber_cement | vinyl | metal | brick | stone | engineered_wood | composite | null>"
  "eaves_vents": "<addressed | not_addressed | null>",
  "vegetation_zone0": "<fully_compliant | partial | non_compliant | null>",
  "vegetation_zone1": "<fully_compliant | partial | non_compliant | null>",
  "fence_gates": "<metal | wood | vinyl | chain_link | composite | wrought_iron | aluminum | bamboo | glass | null>"
  "institutional_trust": "<high | medium | low — based on how they talk about the \
fire department, county, inspectors, and the Ember ordinance, or null if not discussed>",
  "compliance_status": "<compliant | in_progress | non_compliant | partial | null>",
  "insurance_status": "<retained | non_renewed | dropped | null>",
  "years_at_property": <integer or null>,
  "firewise_group": "<name of firewise group or null>",
  "firewise_role": "<active_member | passive_member | leader | null>",
  "spending_so_far": "<minimal | moderate | high | null>",
  "anticipated_costs": "<minimal | moderate | high | null>",
  "financial_flexibility": "<high | moderate | low | null>",
  "social_network": "<active | limited | inactive | null>",
  "action_driver": "<list what initiatives could potentially help them take wildfire mitigation actions — \
e.g. grants, resources, discounts, trusted contractors, better guidance — or null if not discussed>",
  "key_motivation": "<list what motivates them to take wildfire mitigation actions — e.g. \
demonstrate compliance, avoid_penalties, protect their property, avoid losing their insurance, \
social_awareness — or null if not discussed>",
  "mitigation": {{
    "approach": "<A prose paragraph describing how they go about mitigation work — \
their working style (DIY vs. hired out, solo vs. with others), how they research and \
learn what to do (YouTube, neighbours, inspectors, etc.), how they sequence or pace \
the work, and any distinctive methods or philosophies they bring to it. \
Use concrete details from the transcript. null if not discussed.>",
    "material_sourcing": "<A prose paragraph describing how they find and acquire materials — \
e.g. salvaged materials, local suppliers, online orders, bulk delivery, Etsy, contractor \
recommendations, cost-saving strategies. null if not discussed.>",
    "completed": "<A prose paragraph describing all mitigation work they have already \
finished — specific actions taken, areas of the property addressed, items installed or \
removed. Be concrete and specific to what the transcript reports. null if not discussed.>",
    "next_priority": "<A prose paragraph describing the mitigation work they plan to \
tackle next — specific projects, areas, or purchases they have mentioned intending to \
do. null if not discussed.>"
  }},
  "seed_narrative": "<A 3-5 sentence, multi-line paragraph written in second person ('You are...') \
that captures the agent's core story, motivations, and current situation regarding wildfire mitigation. \
Describe who they are, what drives their decisions, how they feel about fire risk, and where they currently \
stand on compliance or action.>",
  "memory_seeds": [
    {{
      "description": "<specific first-person memory grounded in the transcript — past tense, \
concrete detail, e.g. 'I built a brick patio using salvaged bricks'>",
      "importance": <integer 1-10. Use this scale: 9-10 = defining life/property event \
(e.g. near-evacuation, insurance dropped, major fire nearby); 7-8 = significant action taken or \
strong emotional reaction; 5-6 = meaningful conversation or decision about mitigation; \
3-4 = routine observation or minor step taken; 1-2 = passing remark or peripheral detail>,
      "type": "<observation | reflection | conversation>",
      "source": "<direct_experience | secondhand_community | secondhand_media | institutional — \
direct_experience = something they personally witnessed or did; secondhand_community = heard from neighbours, \
friends, or local groups; secondhand_media = learned from news, social media, or publications; \
institutional = from fire dept, county, inspectors, or official notices>"
    }},
    {{
      "description": "<another memory>",
      "importance": <integer 1-10, following the scale above>,
      "type": "<observation | reflection | conversation>",
      "source": "<direct_experience | secondhand_community | secondhand_media | institutional>"
    }},
    {{
      "description": "<another memory>",
      "importance": <integer 1-10, following the scale above>,
      "type": "<observation | reflection | conversation>",
      "source": "<direct_experience | secondhand_community | secondhand_media | institutional>"
    }},
    {{
      "description": "<another memory>",
      "importance": <integer 1-10, following the scale above>,
      "type": "<observation | reflection | conversation>",
      "source": "<direct_experience | secondhand_community | secondhand_media | institutional>"
    }},
    {{
      "description": "<another memory>",
      "importance": <integer 1-10, following the scale above>,
      "type": "<observation | reflection | conversation>",
      "source": "<direct_experience | secondhand_community | secondhand_media | institutional>"
    }},
    {{
      "description": "<another memory>",
      "importance": <integer 1-10, following the scale above>,
      "type": "<observation | reflection | conversation>",
      "source": "<direct_experience | secondhand_community | secondhand_media | institutional>"
    }},
    {{
      "description": "<another memory>",
      "importance": <integer 1-10, following the scale above>,
      "type": "<observation | reflection | conversation>",
      "source": "<direct_experience | secondhand_community | secondhand_media | institutional>"
    }},
    {{
      "description": "<another memory>",
      "importance": <integer 1-10, following the scale above>,
      "type": "<observation | reflection | conversation>",
      "source": "<direct_experience | secondhand_community | secondhand_media | institutional>"
    }},
    {{
      "description": "<another memory>",
      "importance": <integer 1-10, following the scale above>,
      "type": "<observation | reflection | conversation>",
      "source": "<direct_experience | secondhand_community | secondhand_media | institutional>"
    }},
    {{
      "description": "<another memory>",
      "importance": <integer 1-10, following the scale above>,
      "type": "<observation | reflection | conversation>",
      "source": "<direct_experience | secondhand_community | secondhand_media | institutional>"
    }}
  ],
  "held_out_responses": {{
    "insurance_non_renewal": {{
      "real_response": "<Direct quote or close paraphrase from the transcript showing how they \
actually responded to or spoke about insurance non-renewal, or null if not discussed.>",
      "context": "<Brief context for this response — what prompted it in the interview, or null if not discussed.>"
    }},
    "official_regulatory_notice": {{
      "real_response": "<Direct quote or close paraphrase from the transcript showing how they \
actually responded to or spoke about receiving official notices or inspections, or null if not discussed.>",
      "context": "<Brief context for this response — what prompted it in the interview, or null if not discussed.>"
    }}
  }},
  "key_concerns": ["<concern 1>", "<concern 2>", "<concern 3>"],
  "notable_quotes": [
    "<short verbatim quote that captures their voice — under 20 words>",
    "<another quote>"
  ]
}}

Rules:
- memory_seeds MUST be exactly 10 items, specific to THIS person, not generic
- Assign importance scores using the 1-10 scale defined above; aim for a spread across the range rather than clustering all seeds at high importance
- All memory_seeds must be directly grounded in the transcript; do NOT invent or generalise
- held_out_responses must use direct quotes or very close paraphrases from the transcript, not invented reactions
- mitigation sub-fields should be rich prose drawn strictly from the transcript; do not invent details
- display_name must be a pseudonym, never use the interviewee's real name
- persona must reflect actual personality from the interview, not a generic homeowner
- seed_narrative must be written in second person, starting with "You are..."
- If a field is not discussed or cannot be inferred from the transcript, use null
- Respond with JSON only — no explanation, no markdown fences
"""

In [8]:
# Call Gemini to extract profile
def extract_agent_profile(model_client: genai.GenerativeModel, transcript: str, name: str) -> tuple:
    # Send transcript to Gemini and return (parsed_dict, raw_text)
    # Truncate very long transcripts (~6000 words) to stay within context budget
    words = transcript.split()
    if len(words) > 6000:
        transcript = ' '.join(words[:6000]) + "\n\n[transcript truncated for length]"

    prompt = extraction_prompt.format(transcript=transcript)

    response = model_client.generate_content(prompt)

    raw_text = response.text.strip()

    # Strip markdown fences if the model adds them despite instructions
    raw_text = re.sub(r'^```(?:json)?\s*', '', raw_text)
    raw_text = re.sub(r'\s*```$', '', raw_text).strip()

    try:
        profile = json.loads(raw_text)
    except json.JSONDecodeError as e:
        print(f"JSON parse error for {name}: {e}")
        profile = {"agent_id": name.lower(), "parse_error": str(e), "raw": raw_text}

    return profile, raw_text

In [9]:
# Convert profile to agents.yaml format
def profile_to_agent_entry(profile: dict) -> dict:
    # Reshape extracted profile into the agents.yaml schema
    if "parse_error" in profile:
        return {"id": profile.get("agent_id", "unknown"), "error": profile["parse_error"]}

    return {
        "id":                      profile.get("agent_id", "unknown"),
        "display_name":            profile.get("display_name"), # Added from prompt
        "age":                     profile.get("age"),
        "occupation":              profile.get("occupation"), # Added from prompt
        "marital_status":          profile.get("marital_status"), # Added from prompt
        "tenure_type":             profile.get("tenure_type"), # Added from prompt
        "location":                profile.get("location"),
        "persona":                 profile.get("persona", ""),
        "risk_zone":               profile.get("risk_zone", "unknown"),
        "tilden_proximity":        profile.get("tilden_proximity"),
        "property_type":           profile.get("property_type", "unknown"),
        "lot_size":                profile.get("lot_size"),
        "roof_type":               profile.get("roof_type"),
        "siding":                  profile.get("siding"),
        "eaves_vents":             profile.get("eaves_vents"),
        "vegetation_zone0":        profile.get("vegetation_zone0"),
        "vegetation_zone1":        profile.get("vegetation_zone1"),
        "fence_gates":             profile.get("fence_gates"),
        "institutional_trust":     profile.get("institutional_trust", "medium"),
        "compliance_status":       profile.get("compliance_status", "unknown"),
        "insurance_status":        profile.get("insurance_status"),
        "years_at_property":       profile.get("years_at_property"),
        "firewise_group":          profile.get("firewise_group"),
        "firewise_role":           profile.get("firewise_role"),
        "spending_so_far":         profile.get("spending_so_far"),
        "anticipated_costs":       profile.get("anticipated_costs"),
        "financial_flexibility":   profile.get("financial_flexibility"),
        "social_network":          profile.get("social_network"),
        "action_driver":           profile.get("action_driver"),
        "key_motivation":          profile.get("key_motivation"),
        "diy_orientation":         profile.get("diy_orientation"),
        "information_seeking":     profile.get("information_seeking"),
        "emotional_style":         profile.get("emotional_style"),
        "communication_preferences": profile.get("communication_preferences", "social"),
        "mitigation":              profile.get("mitigation", {}), # Added from prompt (nested object)
        "seed_narrative":          profile.get("seed_narrative", ""),
        "memory_seeds":            profile.get("memory_seeds", []),
        "held_out_responses":      profile.get("held_out_responses", {}),
        "key_concerns":            profile.get("key_concerns", []),
        "notable_quotes":          profile.get("notable_quotes", []),
    }


print("Helper functions defined")


Helper functions defined


## Run Extraction (All Transcripts)

In [10]:
# Setup
gemini_model_client = genai.GenerativeModel(model) # Initialize Gemini model

transcript_dir = Path(transcripts_folder_path)
output_dir = Path(output_folder_path)
output_dir.mkdir(parents=True, exist_ok=True)

docx_files = sorted(transcript_dir.glob("*.docx"))
print(f"Processing {len(docx_files)} transcripts...\n")
print("=" * 60)

agents = []
debug_entries = []
errors = []

# Main extraction loop
for i, docx_path in enumerate(docx_files, 1):
    # Derive a short name from the filename
    stem = docx_path.stem
    name = stem.split(" - ")[0].split("_")[0].strip()

    print(f"[{i}/{len(docx_files)}] {name}  ({docx_path.name})")

    try:
        # Extract raw text
        full_text = extract_text_from_docx(docx_path)
        word_count = len(full_text.split())
        print(f"         {word_count:,} words extracted")

        # Filter to homeowner voice only
        homeowner_text = filter_to_interviewee_only(full_text, interviewer_name)
        filtered_wc = len(homeowner_text.split())
        if filtered_wc < word_count:
            print(f"         → filtered to {filtered_wc:,} words (interviewer turns removed)")

        # Call Gemini
        profile, raw_response = extract_agent_profile(gemini_model_client, homeowner_text, name)

        # Convert to agents.yaml format
        agent_entry = profile_to_agent_entry(profile)
        agents.append(agent_entry)

        # Save debug info
        debug_entries.append({
            "file": docx_path.name,
            "name": name,
            "word_count_full": word_count,
            "word_count_filtered": filtered_wc,
            "raw_llm_response": raw_response,
            "parsed_profile": profile
        })

        # Print summary row
        print(f"         agent_id={agent_entry.get('id')} | "
              f"risk={agent_entry.get('risk_zone')} | "
              f"trust={agent_entry.get('institutional_trust')} | "
              f"compliance={agent_entry.get('compliance_status')}")

    except Exception as e:
        print(f"         ERROR: {e}")
        errors.append({"file": docx_path.name, "error": str(e)})

    print()

print("=" * 60)
print(f"Done. {len(agents)} agents extracted, {len(errors)} errors.")

Processing 11 transcripts...

[1/11] Beth  (Beth - Mar 6 at 7_03 PM.docx)
         7,638 words extracted
         agent_id=beth | risk=high | trust=low | compliance=in_progress

[2/11] Charles  (Charles - Mar 6 at 8_06 PM.docx)
         6,236 words extracted
         agent_id=charles | risk=high | trust=medium | compliance=partial

[3/11] Edward  (Edward - Mar 11 at 2_06 PM.docx)
         1,195 words extracted


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 811.42ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 961.95ms


         agent_id=edward | risk=high | trust=low | compliance=compliant

[4/11] Isabelle G  (Isabelle G - Feb 02 2026, 5_00 PM.docx)
         6,935 words extracted
         → filtered to 4,390 words (interviewer turns removed)
         agent_id=isabelle_g | risk=high | trust=high | compliance=compliant

[5/11] Jennifer  (Jennifer - Feb 25 at 1_03 PM.docx)
         4,352 words extracted
         agent_id=jennifer | risk=high | trust=low | compliance=in_progress

[6/11] Lola  (Lola - Feb 23 at 5_09 PM.docx)
         2,540 words extracted
         agent_id=lola | risk=high | trust=low | compliance=compliant

[7/11] Michel  (Michel_.docx)
         8,264 words extracted
         agent_id=michelle | risk=high | trust=medium | compliance=compliant

[8/11] Ross  (Ross - Feb 25 at 10_21 AM.docx)
         4,410 words extracted
         agent_id=ross | risk=high | trust=low | compliance=in_progress

[9/11] Steven L  (Steven L - Feb 02 2026, 12_00 PM.docx)
         2,819 words extracted
         →

## Save Outputs to Google Drive

In [11]:
# Save agents_extracted.yaml
agents_yaml_path = output_dir / agents_yaml_filename
with open(agents_yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(
        {"agents": agents},
        f,
        default_flow_style=False,
        allow_unicode=True,
        sort_keys=False
    )

print(f"Saved {len(agents)} agents → {agents_yaml_path}")

# Save extraction_debug.jsonl
debug_jsonl_path = output_dir / debug_jsonl_filename

with open(debug_jsonl_path, "w", encoding="utf-8") as f:
    for entry in debug_entries:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Saved debug log    → {debug_jsonl_path}")

if errors:
    print(f"\n{len(errors)} file(s) failed:")
    for e in errors:
        print(f"   • {e['file']}: {e['error']}")

Saved 11 agents → /content/drive/MyDrive/INFO 290: Intro to Gen AI/outputs/agents_extracted.yaml
Saved debug log    → /content/drive/MyDrive/INFO 290: Intro to Gen AI/outputs/extraction_debug.jsonl


Summary Table

In [12]:
import pandas as pd

rows = []
for a in agents:
    rows.append({
        "ID":                      a.get("id", "?"),
        "Agent":                   a.get("display_name", "?"),
        "Age":                     a.get("age", "?"),
        "Occupation":              a.get("occupation", "?"), # New from prompt
        "Marital Status":          a.get("marital_status", "?"), # New from prompt
        "Tenure Type":             a.get("tenure_type", "?"), # New from prompt
        "Location":                a.get("location", "?"),
        "Persona Present":         "Yes" if a.get("persona") else "No",
        "Risk Zone":               a.get("risk_zone", "?"),
        "Tilden Proximity":        a.get("tilden_proximity", "?"),
        "Property Type":           a.get("property_type", "?"),
        "Lot Size":                a.get("lot_size", "?"),
        "Roof Type":               a.get("roof_type", "?"),
        "Siding":                  a.get("siding", "?"),
        "Eaves Vents":             a.get("eaves_vents", "?"),
        "Veg Zone 0":              a.get("vegetation_zone0", "?"),
        "Veg Zone 1":              a.get("vegetation_zone1", "?"),
        "Fence Gates":             a.get("fence_gates", "?"),
        "Trust":                   a.get("institutional_trust", "?"),
        "Compliance":              a.get("compliance_status", "?"),
        "Insurance Status":        a.get("insurance_status", "?"),
        "Years at Property":       a.get("years_at_property", "?"),
        "Firewise Group":          a.get("firewise_group", "?"),
        "Firewise Role":           a.get("firewise_role", "?"),
        "Spending So Far":         a.get("spending_so_far", "?"),
        "Anticipated Costs":       a.get("anticipated_costs", "?"),
        "Financial Flexibility":   a.get("financial_flexibility", "?"),
        "Social Network":          a.get("social_network", "?"),
        "Action Driver":           a.get("action_driver", "?"),
        "Key Motivation":          a.get("key_motivation", "?"),
        "DIY Orientation":         a.get("diy_orientation", "?"),
        "Information Seeking":     a.get("information_seeking", "?"),
        "Emotional Style":         a.get("emotional_style", "?"),
        "Comms Channel":           a.get("communication_preferences", "?"),
        "Mitigation Present":      "Yes" if a.get("mitigation", {}) else "No", # New from prompt
        "Narrative Present":       "Yes" if a.get("seed_narrative") else "No",
        "Memory Seeds Count":      len(a.get("memory_seeds", [])),
        "Held Out Responses Present": "Yes" if a.get("held_out_responses", {}) else "No",
        "Key Concerns Count":      len(a.get("key_concerns", [])),
        "Notable Quotes Count":    len(a.get("notable_quotes", [])),
    })

df = pd.DataFrame(rows)
print("\nEXTRACTED AGENT SUMMARY")
display(df)

# Distribution checks — useful for ensuring variance across agent pool
print("\nDISTRIBUTIONS")
for col in ["ID", "Agent", "Age", "Occupation", "Marital Status", "Tenure Type", "Location", "Persona Present", "Risk Zone", "Tilden Proximity",
            "Property Type", "Lot Size", "Roof Type", "Siding", "Eaves Vents", "Veg Zone 0",
            "Veg Zone 1", "Fence Gates", "Trust", "Compliance", "Insurance Status", "Years at Property",
            "Firewise Group", "Firewise Role", "Spending So Far", "Anticipated Costs", "Financial Flexibility",
            "Social Network", "Action Driver", "Key Motivation", "DIY Orientation", "Information Seeking",
            "Emotional Style", "Comms Channel", "Mitigation Present", "Narrative Present",
            "Memory Seeds Count", "Held Out Responses Present", "Key Concerns Count", "Notable Quotes Count"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts().to_string())



EXTRACTED AGENT SUMMARY


,ID,Agent,Age,Occupation,Marital Status,Tenure Type,Location,Persona Present,Risk Zone,Tilden Proximity,...,DIY Orientation,Information Seeking,Emotional Style,Comms Channel,Mitigation Present,Narrative Present,Memory Seeds Count,Held Out Responses Present,Key Concerns Count,Notable Quotes Count
0,beth,Eleanor,65+,Lawyer,married,homeowner,"Berkeley Hills (near Willard Park, top of the ...",Yes,high,front_line,...,None,None,None,social,Yes,Yes,10,Yes,5,2
1,charles,Charles,None,"Former consultant to insurance companies, work...",partnered,homeowner,"Preston Road, Berkeley Hills (top of the hill)",Yes,high,front_line,...,None,None,None,social,Yes,Yes,10,Yes,5,2
2,edward,Arthur,65,None,None,homeowner,Berkeley Hills (Berkeley Woods Firewise area),Yes,high,front_line,...,None,None,None,social,Yes,Yes,10,Yes,3,4
3,isabelle_g,Eleanor,60s,None,partnered,homeowner,Berkeley Hills (Berkeley Woods Firewise area),Yes,high,front_line,...,None,None,None,social,Yes,Yes,10,Yes,5,4
4,jennifer,Margaret,None,None,married,homeowner,Berkeley Hills (near Park Hills Firewise Commu...,Yes,high,mid_hills,...,None,None,None,social,Yes,Yes,10,Yes,7,3
5,lola,Vivian,65+,None,None,homeowner,Berkeley Hills (Wildcat Canyon),Yes,high,front_line,...,None,None,None,social,Yes,Yes,10,Yes,6,4
6,michelle,Michael,None,Scientist / Community Organizer,None,homeowner,Berkeley Hills (Creston - very center of the r...,Yes,high,front_line,...,None,None,None,social,Yes,Yes,10,Yes,3,2
7,ross,David,None,None,None,homeowner,"Berkeley Hills (Wildcat Canyon Road, next to T...",Yes,high,front_line,...,None,None,None,social,Yes,Yes,10,Yes,3,2
8,steven,Robert,None,None,partnered,homeowner,Berkeley Hills (Berkeley Woods Firewise area),Yes,high,front_line,...,None,None,None,social,Yes,Yes,10,Yes,3,2
9,susan,Eleanor,None,retired city council member,None,homeowner,Northeast Berkeley Hills,Yes,high,front_line,...,None,None,None,social,Yes,Yes,10,Yes,5,5



DISTRIBUTIONS

ID:
ID
beth          1
charles       1
edward        1
isabelle_g    1
jennifer      1
lola          1
michelle      1
ross          1
steven        1
susan         1
yen           1

Agent:
Agent
Eleanor     3
Charles     1
Arthur      1
Margaret    1
Vivian      1
Michael     1
David       1
Robert      1
Linh        1

Age:
Age
65+        2
65         1
60s        1
40s-50s    1

Occupation:
Occupation
Lawyer                                                                1
Former consultant to insurance companies, works in fire risk field    1
Scientist / Community Organizer                                       1
retired city council member                                           1
attorney                                                              1

Marital Status:
Marital Status
married      3
partnered    3

Tenure Type:
Tenure Type
homeowner    11

Location:
Location
Berkeley Hills (Berkeley Woods Firewise area)           4
Berkeley Hills (near Willard Park

##Inspect Individual Agent Profile

In [13]:
# Change this to the agent_id you want to inspect
inspect_agent = "jennifer"

match = next((a for a in agents if a.get("id") == inspect_agent), None)

if match is None:
    print(f"Agent '{inspect_agent}' not found. Available: {[a.get('id') for a in agents]}")
else:
    print(f"=== {match['display_name'].upper()} ===")
    print(f"\nPERSONA:")
    print(f"  {match['persona']}")

    print(f"\nKEY ATTRIBUTES:")
    print(f"  age                  : {match.get('age')}")
    print(f"  occupation           : {match.get('occupation')}")
    print(f"  marital_status       : {match.get('marital_status')}")
    print(f"  tenure_type          : {match.get('tenure_type')}")
    print(f"  location             : {match.get('location')}")
    print(f"  risk_zone            : {match.get('risk_zone')}")
    print(f"  tilden_proximity     : {match.get('tilden_proximity')}")
    print(f"  property_type        : {match.get('property_type')}")
    print(f"  lot_size             : {match.get('lot_size')}")
    print(f"  roof_type            : {match.get('roof_type')}")
    print(f"  siding               : {match.get('siding')}")
    print(f"  eaves_vents          : {match.get('eaves_vents')}")
    print(f"  vegetation_zone0     : {match.get('vegetation_zone0')}")
    print(f"  vegetation_zone1     : {match.get('vegetation_zone1')}")
    print(f"  fence_gates          : {match.get('fence_gates')}")
    print(f"  institutional_trust  : {match.get('institutional_trust')}")
    print(f"  compliance_status    : {match.get('compliance_status')}")
    print(f"  insurance_status     : {match.get('insurance_status')}")
    print(f"  years_at_property    : {match.get('years_at_property')}")
    print(f"  firewise_group       : {match.get('firewise_group')}")
    print(f"  firewise_role        : {match.get('firewise_role')}")
    print(f"  spending_so_far      : {match.get('spending_so_far')}")
    print(f"  anticipated_costs    : {match.get('anticipated_costs')}")
    print(f"  financial_flexibility: {match.get('financial_flexibility')}")
    print(f"  social_network       : {match.get('social_network')}")
    print(f"  action_driver        : {match.get('action_driver')}")
    print(f"  key_motivation       : {match.get('key_motivation')}")
    print(f"  diy_orientation      : {match.get('diy_orientation')}")
    print(f"  information_seeking  : {match.get('information_seeking')}")
    print(f"  emotional_style      : {match.get('emotional_style')}")
    print(f"  communication_preferences: {match.get('communication_preferences')}")

    print(f"\nMITIGATION:")
    mitigation = match.get('mitigation', {})
    print(f"  approach           : {mitigation.get('approach')}")
    print(f"  material_sourcing  : {mitigation.get('material_sourcing')}")
    print(f"  completed          : {mitigation.get('completed')}")
    print(f"  next_priority      : {mitigation.get('next_priority')}")

    print(f"\nSEED NARRATIVE:")
    print(f"  {match.get('seed_narrative')}")

    print(f"\nSEED MEMORIES:")
    for i, mem in enumerate(match.get('memory_seeds', []), 1):
        print(f"  {i}. {mem.get('description', 'N/A')}")

    print(f"\nHELD OUT RESPONSES:")
    held_out = match.get('held_out_responses', {})
    if held_out.get('insurance_non_renewal'):
        print(f"  Insurance Non-Renewal: ")
        print(f"    Real Response: {held_out['insurance_non_renewal'].get('real_response')}")
        print(f"    Context: {held_out['insurance_non_renewal'].get('context')}")
    if held_out.get('official_regulatory_notice'):
        print(f"  Official Regulatory Notice: ")
        print(f"    Real Response: {held_out['official_regulatory_notice'].get('real_response')}")
        print(f"    Context: {held_out['official_regulatory_notice'].get('context')}")

    print(f"\nKEY CONCERNS:")
    for concern in match.get('key_concerns', []):
        print(f"  • {concern}")

    print(f"\nNOTABLE QUOTES:")
    for quote in match.get('notable_quotes', []):
        print(f'  "{quote}"')


=== MARGARET ===

PERSONA:
  Margaret is a pragmatic and active homeowner in the Berkeley Hills, deeply engaged in wildfire mitigation efforts alongside her 'handy husband'. She's empathetic to the community's struggles, particularly around insurance and contractor access, and is a leader in sharing information. While resourceful and keen on DIY to manage costs and aesthetics, she's also frustrated by the lack of clear, applicable guidance for unique property configurations and the perceived capriciousness of institutional support, including insurance companies and fire department programs.

KEY ATTRIBUTES:
  age                  : None
  occupation           : None
  marital_status       : married
  tenure_type          : homeowner
  location             : Berkeley Hills (near Park Hills Firewise Community)
  risk_zone            : high
  tilden_proximity     : mid_hills
  property_type        : single_family
  lot_size             : None
  roof_type            : None
  siding        

## Preview agents.yaml Output

In [14]:
# Print the first 2 agents from the YAML to verify formatting
preview = {"agents": agents[:2]}
print(yaml.dump(preview, default_flow_style=False, allow_unicode=True, sort_keys=False))

agents:
- id: beth
  display_name: Eleanor
  age: 65+
  occupation: Lawyer
  marital_status: married
  tenure_type: homeowner
  location: Berkeley Hills (near Willard Park, top of the hills)
  persona: Eleanor is a working lawyer residing in the Berkeley Hills, having previously
    been a graduate student at Berkeley for many years. She is a pragmatic and compliant
    homeowner, willing to invest significant time and money into wildfire mitigation,
    but deeply frustrated by the inconsistencies in official guidance and the perceived
    inaction of city and state entities. She values her time more than money and relies
    on trusted social networks for contractor recommendations, even if it means potentially
    overpaying. Despite her efforts, she remains critical of the bureaucratic hurdles
    and the uneven distribution of responsibility for wildfire safety.
  risk_zone: high
  tilden_proximity: front_line
  property_type: single_family
  lot_size: null
  roof_type: null
  sid

## Re-run a Single Agent

Use this if one extraction failed or produced a poor result. It re-runs just that one file without re-processing the others.

In [15]:
# Set to the filename you want to re-run (just the filename, not the full path)
rerun_file = "Ross - Feb 25 at 10_21 AM.docx"

rerun_path = transcript_dir / rerun_file

if not rerun_path.exists():
    print(f"File not found: {rerun_path}")
else:
    name = rerun_file.split(" - ")[0].split("_")[0].strip()
    print(f"Re-running extraction for: {name}")

    full_text = extract_text_from_docx(rerun_path)
    homeowner_text = filter_to_interviewee_only(full_text, interviewer_name)
    profile, raw_response = extract_agent_profile(gemini_model_client, homeowner_text, name)
    agent_entry = profile_to_agent_entry(profile)

    # Replace in agents list if already exists, otherwise append
    existing_ids = [a.get("id") for a in agents]
    if agent_entry.get("id") in existing_ids:
        idx = existing_ids.index(agent_entry.get("id"))
        agents[idx] = agent_entry
        print(f"Replaced existing entry for '{agent_entry.get('id')}'")
    else:
        agents.append(agent_entry)
        print(f"Added new entry for '{agent_entry.get('id')}'")

    print(f"\nResult:")
    print(json.dumps(agent_entry, indent=2))

    # Optionally re-save the YAML
    with open(agents_yaml_path, "w", encoding="utf-8") as f:
        yaml.dump({"agents": agents}, f, default_flow_style=False,
                  allow_unicode=True, sort_keys=False)
    print(f"\nUpdated {agents_yaml_path}")

Re-running extraction for: Ross
Replaced existing entry for 'ross'

Result:
{
  "id": "ross",
  "display_name": "David",
  "age": null,
  "occupation": null,
  "marital_status": null,
  "tenure_type": "homeowner",
  "location": "Berkeley Hills (Wildcat Canyon Road, near Tilden Park)",
  "persona": "David is a proactive and financially capable homeowner who has lived in the Berkeley Hills for about five years. He takes wildfire risk very seriously, believing it's a matter of \"when, not if,\" and is highly motivated to protect his property. While committed to complying with the Ember ordinance, he is deeply frustrated by the lack of clear, consistent guidance from fire inspectors and the difficulty in interpreting the rules, often feeling like he's navigating a \"tricky\" and ambiguous system. Despite these challenges, he maintains a pragmatic and optimistic outlook, keen to collaborate with experts to ensure his home and community are better prepared.",
  "risk_zone": "high",
  "tilden